# Stage 1 zero-overlap DPO for BioASQ factoid QA (Qwen2.5-0.5B)

This configuration runs a fresh Stage 1 experiment from the original 0.5B SFT adapter using only strict-equivalence `C3 > C1` pairs whose weak-normalized word-set Jaccard score is exactly zero.

- **Stage 1 data:** 398 pairs across 176 questions
- **Filter:** case-fold and collapse whitespace, preserve punctuation, then require `Jaccard(C3, C1) = 0`
- **Objective:** standard DPO
- **Training/evaluation context:** 4,096 tokens
- **Checkpoint selection:** official BioASQ dev MRR
- **Output isolation:** a new run directory, leaving previous experiments unchanged

The staged dataset root also contains unchanged Stage 2 and Stage 3 files, but this notebook is currently configured to stop after Stage 1.


In [ ]:
from pathlib import Path
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import sys, json, importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
sys.path.insert(0, str(PROJECT_ROOT))

import cse_dpo.train_factoid_three_stage_dpo as three_stage_dpo
three_stage_dpo = importlib.reload(three_stage_dpo)
Config = three_stage_dpo.Config
STAGES = three_stage_dpo.STAGES
SUPPORTED_OPTIMIZERS = three_stage_dpo.SUPPORTED_OPTIMIZERS
SUPPORTED_OBJECTIVES = three_stage_dpo.SUPPORTED_OBJECTIVES
normalize_optimizer_name = three_stage_dpo._normalize_optimizer_name
MODEL_PRESETS = three_stage_dpo.MODEL_PRESETS
STAGED_DATASET_PRESETS = three_stage_dpo.STAGED_DATASET_PRESETS
load_stage_rows = three_stage_dpo.load_stage_rows
run_three_stage = three_stage_dpo.run_three_stage

# Main controls
SMOKE_TEST = False
SMOKE_PAIRS = 8
MODEL_PRESET = "qwen25_05b"
DATASET_PRESET = "gold_supported_strict_equivalence_v2_1130_stage1_jaccard0"

# Use "concept_learning" to run only stage 1 first. Set None later to continue stages 2 and 3.
STOP_AFTER_STAGE = "concept_learning"  # Train Stage 1 only.
# To reuse stage 1 and train stage 3 directly from it, skip the fragile stage 2.
SKIP_STAGES = []

# Optional: keep the same output_root but write retry stages to sibling folders.
# Example: {"format_alignment": "retry_beta003_lr1e7"} writes to
# stage_2_format_alignment_retry_beta003_lr1e7 instead of overwriting/reusing stage_2_format_alignment.
STAGE_OUTPUT_SUFFIXES = {}
# Stage 2 changes checkpoint selection to retention_score, which changes the run fingerprint.
# This override deliberately reuses the already completed Stage 1 best checkpoint.
ALLOW_MIXED_OBJECTIVE_RESUME = False

RUN_TAG = "smoke" if SMOKE_TEST else "full"
OPTIMIZER = "AdamW"  # Options: RMSprop, AdamW, Adam, SGD, Adafactor, paged_adamw_8bit, adamw_8bit.
WARMUP_FRACTION = 0.05  # Fraction of optimizer updates used for LR warmup; set 0.0 to disable.
OBJECTIVE = "dpo"  # Options: dpo, apo_zero, apo_down, dpo_adaptive_nll, cal_dpo.
ADAPTIVE_NLL_WEIGHT = 0.02  # Controlled Stage 2 ablation; active only for dpo_adaptive_nll.
LORA_DROPOUT = 0.05  # Set None to keep the loaded adapter config; try 0.10 for stronger regularization.
APPEND_EOS_TO_COMPLETIONS = True
ALLOW_TRUNCATED_EXAMPLES = False
DROP_TRUNCATED_EXAMPLES = True
# Shared generated-dev protocol; keep this identical to the SFT evaluator.
STANDARD_EVAL_MAX_SEQ_LENGTH = 4096
STANDARD_EVAL_MAX_NEW_TOKENS = 64

# Training-only safety budget for backward. None means use every encoded pair.
# The 0.5B model fits the configured 4096-token training budget on a 16 GB GPU.
TRAIN_BACKPROP_MAX_LENGTH = 4096
BACKPROP_TAG = "bpall" if TRAIN_BACKPROP_MAX_LENGTH is None else f"bp{TRAIN_BACKPROP_MAX_LENGTH}"

# Zero-overlap strict-equivalence set: Stage 1 has 398 C3>C1 pairs across 176 questions.
ACTIVE_STAGED_ROOT = Path(STAGED_DATASET_PRESETS[DATASET_PRESET])
STAGE1_DATA = ACTIVE_STAGED_ROOT / dict(STAGES)["concept_learning"]
STAGE2_DATA = ACTIVE_STAGED_ROOT / dict(STAGES)["format_alignment"]

RESUME_OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_05b_strict_equivalence_v2_1130_stage1_jaccard0_standard_dpo_bp4096"

# Optional semantic judge for generated-dev evaluation. Concept learning can select by semantic accuracy if enabled.
SEMANTIC_JUDGE_ENABLED = False
SEMANTIC_JUDGE_MODEL = os.environ.get("FACTOID_SEMANTIC_JUDGE_MODEL", "gpt-4.1-mini-2025-04-14")
SEMANTIC_JUDGE_ENDPOINT = os.environ.get("FACTOID_SEMANTIC_JUDGE_ENDPOINT", "https://api.openai.com/v1/chat/completions")
SEMANTIC_JUDGE_API_KEY_FILE = PROJECT_ROOT / "open_ai_api.txt"
SEMANTIC_JUDGE_CACHE_ROOT = PROJECT_ROOT / "Artifacts/Factoid_SFT/class_judgments/three_stage_dpo_generated_dev_semantic_cache"
SEMANTIC_JUDGE_MAX_NEW_CALLS = None  # Use an int for API-budgeted smoke runs.

assert normalize_optimizer_name(OPTIMIZER) in SUPPORTED_OPTIMIZERS, sorted(SUPPORTED_OPTIMIZERS)
assert OBJECTIVE in SUPPORTED_OBJECTIVES, sorted(SUPPORTED_OBJECTIVES)
CFG = Config(
    smoke_test=SMOKE_TEST,
    smoke_pairs=SMOKE_PAIRS,
    stop_after_stage=STOP_AFTER_STAGE,
    skip_stages=SKIP_STAGES,
    stage_output_suffixes=STAGE_OUTPUT_SUFFIXES,
    allow_mixed_objective_resume=ALLOW_MIXED_OBJECTIVE_RESUME,
    model_preset=MODEL_PRESET,
    dataset_preset=DATASET_PRESET,
    staged_root=str(ACTIVE_STAGED_ROOT),
    output_root=str(RESUME_OUTPUT_ROOT),
    epochs=8,
    objective=OBJECTIVE,
    adaptive_nll_weight=ADAPTIVE_NLL_WEIGHT,
    optimizer=OPTIMIZER,
    warmup_fraction=WARMUP_FRACTION,
    early_stopping_patience=3,
    lora_dropout=LORA_DROPOUT,
    append_eos_to_completions=APPEND_EOS_TO_COMPLETIONS,
    allow_truncated_examples=ALLOW_TRUNCATED_EXAMPLES,
    drop_truncated_examples=DROP_TRUNCATED_EXAMPLES,
    train_backprop_max_length=TRAIN_BACKPROP_MAX_LENGTH,
    generated_eval_max_seq_length=STANDARD_EVAL_MAX_SEQ_LENGTH,
    generated_eval_max_new_tokens=STANDARD_EVAL_MAX_NEW_TOKENS,
    dev_eval_questions=8 if SMOKE_TEST else None,
    semantic_judge_enabled=SEMANTIC_JUDGE_ENABLED,
    semantic_judge_model=SEMANTIC_JUDGE_MODEL,
    semantic_judge_endpoint=SEMANTIC_JUDGE_ENDPOINT,
    semantic_judge_api_key_file=str(SEMANTIC_JUDGE_API_KEY_FILE),
    semantic_judge_cache_root=str(SEMANTIC_JUDGE_CACHE_ROOT),
    semantic_judge_max_new_calls=SEMANTIC_JUDGE_MAX_NEW_CALLS,
    stage_settings={
        "concept_learning": {"epochs": 8, "learning_rate": 2e-6, "beta": 0.05, "objective": "dpo", "adaptive_nll_weight": ADAPTIVE_NLL_WEIGHT, "batch_size": 1, "gradient_accumulation": 8, "max_length": 4096, "optimizer": OPTIMIZER, "warmup_fraction": WARMUP_FRACTION, "weight_decay": 0.01, "eval_every_updates": 0, "early_stopping_patience": 3, "selection_metric": "dev_mrr", "semantic_judge_enabled": SEMANTIC_JUDGE_ENABLED, "lora_dropout": LORA_DROPOUT, "append_eos_to_completions": APPEND_EOS_TO_COMPLETIONS, "allow_truncated_examples": ALLOW_TRUNCATED_EXAMPLES, "drop_truncated_examples": DROP_TRUNCATED_EXAMPLES, "train_backprop_max_length": TRAIN_BACKPROP_MAX_LENGTH},
        "format_alignment": {"epochs": 4, "learning_rate": 5e-7, "beta": 0.3, "objective": "dpo_adaptive_nll", "adaptive_nll_weight": ADAPTIVE_NLL_WEIGHT, "batch_size": 1, "gradient_accumulation": 16, "max_length": 4096, "optimizer": OPTIMIZER, "warmup_fraction": WARMUP_FRACTION, "weight_decay": 0.05, "eval_every_updates": 25, "early_stopping_patience": 10, "selection_metric": "retention_score", "retention_min_rate": 0.98, "retention_loss_penalty": 2.0, "retention_anchor_weight": 0.0, "semantic_judge_enabled": False, "lora_dropout": LORA_DROPOUT, "append_eos_to_completions": APPEND_EOS_TO_COMPLETIONS, "allow_truncated_examples": ALLOW_TRUNCATED_EXAMPLES, "drop_truncated_examples": DROP_TRUNCATED_EXAMPLES, "train_backprop_max_length": TRAIN_BACKPROP_MAX_LENGTH},
        "hierarchical_ranking": {"epochs": 8, "learning_rate": 1e-6, "beta": 0.05, "objective": "dpo", "adaptive_nll_weight": ADAPTIVE_NLL_WEIGHT, "batch_size": 1, "gradient_accumulation": 8, "max_length": 4096, "optimizer": OPTIMIZER, "warmup_fraction": WARMUP_FRACTION, "weight_decay": 0.01, "eval_every_updates": 0, "early_stopping_patience": 3, "selection_metric": "dev_mrr", "semantic_judge_enabled": False, "lora_dropout": LORA_DROPOUT, "append_eos_to_completions": APPEND_EOS_TO_COMPLETIONS, "allow_truncated_examples": ALLOW_TRUNCATED_EXAMPLES, "drop_truncated_examples": DROP_TRUNCATED_EXAMPLES, "train_backprop_max_length": TRAIN_BACKPROP_MAX_LENGTH},
    },
    batch_size=1,
    gradient_accumulation=8,
    max_length=4096,
)

print(json.dumps({
    "model_preset": CFG.model_preset,
    "dataset_preset": CFG.dataset_preset,
    "available_presets": sorted(MODEL_PRESETS),
    "available_dataset_presets": sorted(STAGED_DATASET_PRESETS),
    "available_optimizers": sorted(SUPPORTED_OPTIMIZERS),
    "staged_root": CFG.staged_root,
    "active_staged_root": str(ACTIVE_STAGED_ROOT),
    "stage1_data": str(STAGE1_DATA),
    "stage2_data": str(STAGE2_DATA),
    "base_model": CFG.base_model,
    "initial_adapter": CFG.initial_adapter,
    "output_root": CFG.output_root,
    "resume_output_root": str(RESUME_OUTPUT_ROOT),
    "smoke_test": CFG.smoke_test,
    "stop_after_stage": CFG.stop_after_stage,
    "skip_stages": CFG.skip_stages,
    "allow_mixed_objective_resume": CFG.allow_mixed_objective_resume,
    "optimizer": OPTIMIZER,
    "warmup_fraction": CFG.warmup_fraction,
    "objective": CFG.objective,
    "adaptive_nll_weight": CFG.adaptive_nll_weight,
    "lora_dropout": CFG.lora_dropout,
    "append_eos_to_completions": CFG.append_eos_to_completions,
    "allow_truncated_examples": CFG.allow_truncated_examples,
    "drop_truncated_examples": CFG.drop_truncated_examples,
    "train_backprop_max_length": CFG.train_backprop_max_length,
    "generated_eval_max_seq_length": CFG.generated_eval_max_seq_length,
    "generated_eval_max_new_tokens": CFG.generated_eval_max_new_tokens,
    "semantic_judge_enabled": CFG.semantic_judge_enabled,
    "semantic_judge_model": CFG.semantic_judge_model,
    "semantic_judge_cache_root": CFG.semantic_judge_cache_root,
    "stage_selection_metrics": {stage: CFG.stage_settings[stage]["selection_metric"] for stage in CFG.stage_settings},
    "stage_objectives": {stage: CFG.stage_settings[stage]["objective"] for stage in CFG.stage_settings},
    "stage_warmup_fractions": {stage: CFG.stage_settings[stage].get("warmup_fraction") for stage in CFG.stage_settings},
    "stage_adaptive_nll_weights": {stage: CFG.stage_settings[stage].get("adaptive_nll_weight") for stage in CFG.stage_settings},
    "stage_files": dict(STAGES),
}, indent=2))


In [ ]:
# Validate the selected Stage 1 data and SFT checkpoint before loading the GPU.
required_paths = {
    "initial_adapter": Path(CFG.initial_adapter),
    "stage1_c3_gt_c1": STAGE1_DATA,
}
for label, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{label}: {path}")
    print(f"{label}: {path}")


In [ ]:
# Inspect stage-pair counts before training.
from collections import Counter
stages = load_stage_rows(CFG)
selected_stage_names = [name for name, _ in STAGES]
if STOP_AFTER_STAGE is not None:
    selected_stage_names = selected_stage_names[:selected_stage_names.index(STOP_AFTER_STAGE) + 1]
summary = {}
for stage, rows in stages.items():
    if stage not in selected_stage_names:
        continue
    summary[stage] = {
        "pairs": len(rows),
        "questions": len({r["question_id"] for r in rows}),
        "pair_classes": dict(Counter(f"{r.get('chosen_class')}>{r.get('rejected_class')}" for r in rows)),
    }
print(json.dumps(summary, indent=2))


In [ ]:
# Run. Completed stages are reused automatically from their manifest.
# If STOP_AFTER_STAGE="concept_learning", this cell trains only stage 1 and exits.
results = run_three_stage(CFG)
print(json.dumps(results, indent=2, default=str))


In [ ]:
# Inspect output artifacts.
from pathlib import Path
out = Path(CFG.output_root)
print("Output root:", out)
for path in sorted(out.glob("**/manifest.json")):
    print("manifest:", path)
for path in sorted(out.glob("**/data_summary.json")):
    print("data summary:", path)
for path in sorted(out.glob("**/dropped_backprop_pairs.json")):
    print("dropped backprop audit:", path)
for path in sorted(out.glob("**/training_curves.png")):
    print("plot:", path)
